**Objetivo:**

Preparar los datos para el entrenamiento de los modelos supervisados: 

-Cargar los datasets (train_* y test).

-Definir variable objetivo y variables predictoras.

-Validar estructura de los datos.

-Calcular pesos de clase (compute_class_weight).

-Guardar la tabla pesos_clases_modelos.

Entrada:
            Bases: 
                    train_original

                    train_balanceado_500

                    train_balanceado_1000

                    train_balanceado_2000

                    test


            Salida:
                    Tabla:

                            pesos_clases_modelos



In [0]:

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.functions import count, lit, col, when, regexp_replace, concat, round
from pyspark.sql.types import LongType

# Manejo de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt

# K-Modes
#from kmodes.kmodes import KModes

# Tiempo de ejecución
import time

# Ignorar advertencias
import warnings
warnings.filterwarnings("ignore")

#sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight
#from imblearn.over_sampling import RandomOverSampler

**Cargar conjuntos de entrenamiento y prueba**

In [0]:
ruta = "ml_proyecto_7405607705157039.default"


train_original = spark.table(
    f"{ruta}.train_original"
)

train_balanceado_500 = spark.table(
    f"{ruta}.train_balanceado_500"
)

train_balanceado_1000 = spark.table(
    f"{ruta}.train_balanceado_1000"
)

train_balanceado_2000 = spark.table(
    f"{ruta}.train_balanceado_2000"
)


test = spark.table(
    f"{ruta}.test"
)

In [0]:
#Validar tamaños

datasets = {
    "original": train_original,
    "balanceado_500": train_balanceado_500,
    "balanceado_1000": train_balanceado_1000,
    "balanceado_2000": train_balanceado_2000,
    "test": test
}


for nombre, df in datasets.items():
    print(nombre, df.count())

**Definir variable objetivo y variables predictoras**

In [0]:
target = "nivel_peligro"

In [0]:
variables_modelo = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",    
    "presunto_agresor_cod"
]



In [0]:
#Separación X e y

X_cols = variables_modelo

y_col = target

In [0]:
#Validación antes del entrenamiento

for nombre, df in datasets.items():
    
    faltantes = (
        set(X_cols + [y_col])
        -
        set(df.columns)
    )
    
    print(nombre)
    
    if len(faltantes) == 0:
        print("OK - Todas las variables están presentes")
    else:
        print("Faltan:", faltantes)

In [0]:
#Ajuste de One-Hot Encoding

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            variables_modelo
        )
    ]
)

**Cálculo de pesos de clase**

El objetivo de esta sección es que cada escenario tenga sus propios pesos de clase. Esto permitirá que los modelos den mayor importancia a las clases minoritarias sin tener que modificar manualmente los valores.

La lógica será:

train_original: tendrá un peso muy alto para Peligro extremo porque tiene pocos registros.

train_balanceado_500, 1000, 2000: tendrán pesos menores porque aumentamos esa clase.

Cada modelo recibirá los pesos correspondientes al escenario con el que fue entrenado.

**Cálculo de pesos de clase**

In [0]:
#Crear función para calcular pesos automáticamente

def calcular_pesos_clase(y):

    clases = np.unique(y)

    pesos = compute_class_weight(
        class_weight="balanced",
        classes=clases,
        y=y
    )

    pesos_dict = dict(
        zip(clases, pesos)
    )

    return pesos_dict

In [0]:
#Calcular pesos para cada escenario

#Primero convertir únicamente la variable objetivo. 

pesos_escenarios = {}

for nombre, df in datasets.items():

    if nombre != "test":

        y = (
            df
            .select(target)
            .toPandas()[target]
        )

        pesos = calcular_pesos_clase(y)

        pesos_escenarios[nombre] = pesos

In [0]:
#Visualizar resultados

pesos_escenarios

**Interpretación de pesos de clase por escenario**

Los pesos se calculan con:

pesoi = N / (K * nᵢ)

N = número total de registros del escenario.
K = número de clases.
nᵢ = número de registros de la clase i.

Por eso:

Las clases frecuentes tienen pesos cercanos a 1.
Las clases poco representadas tienen pesos mayores.
El modelo penaliza más los errores cometidos sobre clases minoritarias.

In [0]:
#Tabla de pesos de clase para modelos

# Crear lista de registros
lista_pesos = []

for escenario, pesos in pesos_escenarios.items():
    
    for clase, peso in pesos.items():
        
        lista_pesos.append(
            {
                "escenario": escenario,
                "clase": clase,
                "peso_clase": float(peso)
            }
        )


# Crear DataFrame Pandas
df_pesos = pd.DataFrame(lista_pesos)

display(df_pesos)

In [0]:
#Convertir a Spark

df_pesos_spark = spark.createDataFrame(df_pesos)

df_pesos_spark.printSchema()

In [0]:
#Guardar como tabla Delta
(
    df_pesos_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.pesos_clases_modelos"
    )
)

In [0]:
df_pesos_guardada = spark.table(
    "ml_proyecto_7405607705157039.default.pesos_clases_modelos"
)

display(df_pesos_guardada)